# GMDA - Full pipeline

## Documentation

The following code propose 3 modes:
* *poly* to define landmarks as polygons only ;
* *line* to define landmarks as junctions only ;
* *both* to use all the geometries.

For the dataset preparation, 2 GeoDataFrames are created:
* *bsm_data* to collect information from the GeoJSON file of the base map ;
* *skm_data* to collect information from the GeoJSON file of the sketch map.

The landmarks detection and matching works with 5 data structures:
* *___bsm_dict_mbr___ { id_geom: [coordinates] }*\
It stores for every landmark of the base map the *(x, y)* coordinates of its 8 MBR vertices, clockwise starting by the top-left corner.
* *___skm_dict_mbr___ { id_geom: [coordinates] }*\
The same for the sketch map.
* *___bsm_junctions___ { id_junc: [ [coordinates], [ids_intersecting_roads] ] }* - Only for *line* and *both* mode\
It stores for every detected junction on the base map its *(x, y)* coordinates, and the identifiers of the roads intersecting it on the base map.
* *___skm_junctions___ { id_junc: [ [coordinates], [ids_intersecting_roads] ] }* - Only for *line* and *both* mode\
The same for the sketch map.
* *___dict_align___ { base_map_id: sketch_map_id }*\
It stores for every landmark drawn on the sketch map the (base map id, sketch map id) pair, only considering 1:1 alignments.


### Algorithm

**Input:** GeoJSON file from SketchMapia, including coordinates and alignment

1. Convert the original GeoJSON file to a GeoDataFrame in order to make the data readable.

2. Identify the landmarks on the base map aligned with another landmark on the sketch map (1:1 alignment).

3. For every landmark, get its geometry on the base map from the GeoDataFrame.

4. If the landmark is a *polygon*, calculate its Minimum Bounding Rectangle and the coordinates of its 8 vertices.\
If the landmark is a *line*, identify intersections by comparing with the other lines. Then, calculate their Minimum Bounding Rectangles (considering a 1-px buffer) and the coordinates of their 8 vertices.

5. Repeat the steps 2 to 5 for the sketch map.

6. Compute the values of *nTL* (total number of landmarks on the base map) and *nDL* (number of aligned landmarks drawn on the sketch map), and then deduct the number of comparisons *n*.

7. Calculate the 6 GMDA measures by exploring all the doable 1:1 landmark comparisons on the sketch map, and their equivalent on the base map.\
For the first landmark, realize comparisons with all the other landmarks. For the second one, with all the other landmarks except the first one, and so on...\
For the last landmark, all comparisons are already done with the previous ones.

8. Store the final results in a CSV file.

**Output:** CSV file with sketch map id, nTL, nDL and the 6 GMDA measures

## Code

In [1]:
import matplotlib.pyplot as plt
import geopandas as gpd
import shapely as shp
import pandas as pd
import numpy as np
import math
import os

# TO CHOOSE ('poly', 'line', 'both')
mode = "line"

# Dataset to store the results
results = pd.DataFrame(columns=['Location', 'XX', 'Skm', 'nTL', 'nDL', 'CanOrg', 'CanAcc', 'ScBias', 'DistAcc', 'RotBias', 'AngAcc'])
pd.set_option('display.max_rows', None)

for loc in [1, 3, 4, 5]:
    for xx in ['AS', 'CM', 'JK']:
        for skm_id in range(1, 11):

            # ---------- PREPARATION OF THE DATASET -----------------------------------------------------------------------------------------------------

            geojson_path = "./datasets/LocationsCorrectedMerged/Location" + str(loc) + "_gen/_location" + str(loc) + "_" + xx + "_Sketch_" + str(skm_id) + ".jpg.geojson"
            if os.path.exists(geojson_path): # if base map image saved as JPG
                bsm_data = gpd.read_file(geojson_path)
            else:
                bsm_data = gpd.read_file("./datasets/LocationsCorrectedMerged/Location" + str(loc) + "_gen/_location" + str(loc) + "_" + xx + "_Sketch_" + str(skm_id) + ".png.geojson")
            skm_data = gpd.read_file("./datasets/LocationsCorrectedMerged/Location" + str(loc) + "_gen/" + xx + "_Sketch_" + str(skm_id) + ".jpg.geojson")
            

            # ---------- LANDMARKS DETECTION ON THE BASE MAP --------------------------------------------------------------------------------------------

            bsm_dict_mbr = {} # stores the MBR of each landmark
            dict_align = {} # stores the (base map id, sketch map id) pairs

            # POLYGONS
            if mode != "line":
                bsm_polys = bsm_data.loc[(bsm_data['otype'] == 'Polygon') & (~(bsm_data['SketchAlign'].map(len, na_action='ignore') > 1))] # subdataset with 1:1 aligned polygons only
                for i in range(bsm_polys.shape[0]): # for each line
                    id_poly = int(bsm_polys[['id']].iloc[i, 0])
                    coords_poly = list(bsm_polys[['geometry']].iloc[i, 0].exterior.coords)
                    x_min, y_min = np.inf, np.inf # starts the MBR computation
                    x_max, y_max = -np.inf, -np.inf
                    for coords in coords_poly:
                        x, y = coords
                        if x < x_min:
                            x_min = x
                        if x > x_max:
                            x_max = x
                        if y < y_min:
                            y_min = y
                        if y > y_max:
                            y_max = y
                    up_left, up_right, down_left, down_right = [x_min, y_max], [x_max, y_max], [x_min, y_min], [x_max, y_min] # defines the 8 vertices
                    up_center = [(x_min + x_max) / 2, y_max]
                    down_center = [(x_min + x_max) / 2, y_min]
                    left_center = [x_min, (y_min + y_max) / 2]
                    right_center = [x_max, (y_min + y_max) / 2]
                    bsm_dict_mbr[id_poly] = np.array([up_left, up_center, up_right, right_center, down_right, down_center, down_left, left_center, up_left])
                    if bsm_polys[['aligned']].iloc[i, 0]:
                        dict_align[id_poly] = int(bsm_polys[['SketchAlign']].iloc[i, 0][0][1:]) # landmark id on sketch map saved without 'S'

            # JUNCTIONS
            if mode != "poly":
                bsm_junctions = {} # stores the coordinates and the intersecting roads ids for each junction
                bsm_lines = bsm_data.loc[(bsm_data['otype'] == 'Line') & (~(bsm_data['SketchAlign'].map(len, na_action='ignore') > 1))] # subdataset with 1:1 aligned lines only
                jb_id = 0 # self-generated id for the junctions on the base map
                for i in range(bsm_lines.shape[0]): # for each line
                    l1_id = int(bsm_lines[['id']].iloc[i, 0])
                    l1_geom = bsm_lines[['geometry']].iloc[i, 0]
                    for j in range(i+1, bsm_lines.shape[0]): # for each other unexplored line
                        l2_id = int(bsm_lines[['id']].iloc[j, 0])
                        l2_geom = bsm_lines[['geometry']].iloc[j, 0]
                        inter_coords = shp.get_coordinates(l1_geom.intersection(l2_geom)).tolist()
                        if inter_coords != []: # if there is really an intersection point
                            new_junc = True # in a first time, suppose that the junction is new
                            for junc in bsm_junctions.keys():
                                if inter_coords[0] == bsm_junctions[junc][0]: # if this junction was already detected
                                    if l1_id in bsm_junctions[junc][1] and not l2_id in bsm_junctions[junc][1]: # add the missing id to the line ids
                                        bsm_junctions[junc][1].append(l2_id)
                                    elif l2_id in bsm_junctions[junc][1] and not l1_id in bsm_junctions[junc][1]:
                                        bsm_junctions[junc][1].append(l1_id)
                                    new_junc = False # finally this junction is already known
                                    continue
                            if new_junc: # if finally this junction is really new
                                bsm_junctions['JB'+str(jb_id)] = [inter_coords[0], [l1_id, l2_id]] # add [JBx]: [[coords], [id_lines]] in bsm_junctions
                                jb_id += 1
                for junc in bsm_junctions.keys():
                    r_buffer = 1 # buffer around the point
                    x, y = bsm_junctions[junc][0]
                    x_min, x_max, y_min, y_max = x - r_buffer, x + r_buffer, y - r_buffer, y + r_buffer # starts the MBR computation
                    up_left, up_right, down_left, down_right = [x_min, y_max], [x_max, y_max], [x_min, y_min], [x_max, y_min]
                    up_center = [(x_min + x_max) / 2, y_max]
                    down_center = [(x_min + x_max) / 2, y_min]
                    left_center = [x_min, (y_min + y_max) / 2]
                    right_center = [x_max, (y_min + y_max) / 2]
                    bsm_dict_mbr[junc] = np.array([up_left, up_center, up_right, right_center, down_right, down_center, down_left, left_center, up_left])
            

            # ---------- LANDMARKS DETECTION ON THE SKETCH MAP ------------------------------------------------------------------------------------------

            skm_dict_mbr = {} # stores the MBR of each landmark

            # POLYGONS
            if mode != "line":
                skm_polys = skm_data.loc[(skm_data['aligned'] == True) & ((skm_data['otype'] == 'Polygon') | (skm_data['otype'] == 'CircleMarker'))] # a polygon can be represented as a CircleMarker in the sketch map GeoJSON file
                skm_polys = skm_polys[skm_polys['id'].isin(dict_align.values())] # considers the aligned polygons only
                for i in range(skm_polys.shape[0]):
                    id_poly = int(skm_polys[['id']].iloc[i, 0])
                    if skm_polys[['otype']].iloc[i, 0] == 'Polygon': # MBR computation if type Polygon
                        coords_poly = list(skm_polys[['geometry']].iloc[i, 0].exterior.coords)
                        x_min, y_min = np.inf, np.inf
                        x_max, y_max = -np.inf, -np.inf
                        for coords in coords_poly:
                            x, y = coords
                            if x < x_min:
                                x_min = x
                            if x > x_max:
                                x_max = x
                            if y < y_min:
                                y_min = y
                            if y > y_max:
                                y_max = y
                        up_left, up_right, down_left, down_right = [x_min, y_max], [x_max, y_max], [x_min, y_min], [x_max, y_min]
                        up_center = [(x_min + x_max) / 2, y_max]
                        down_center = [(x_min + x_max) / 2, y_min]
                        left_center = [x_min, (y_min + y_max) / 2]
                        right_center = [x_max, (y_min + y_max) / 2]
                        skm_dict_mbr[id_poly] = np.array([up_left, up_center, up_right, right_center, down_right, down_center, down_left, left_center, up_left])
                    else: # MBR computation if type CircleMarker
                        r_buffer = 1 # buffer around the point
                        x, y = list(skm_polys[['geometry']].iloc[i, 0].coords)[0] # only 1 point
                        x_min, x_max, y_min, y_max = x - r_buffer, x + r_buffer, y - r_buffer, y + r_buffer
                        up_left, up_right, down_left, down_right = [x_min, y_max], [x_max, y_max], [x_min, y_min], [x_max, y_min]
                        up_center = [(x_min + x_max) / 2, y_max]
                        down_center = [(x_min + x_max) / 2, y_min]
                        left_center = [x_min, (y_min + y_max) / 2]
                        right_center = [x_max, (y_min + y_max) / 2]
                        skm_dict_mbr[id_poly] = np.array([up_left, up_center, up_right, right_center, down_right, down_center, down_left, left_center, up_left])

            # JUNCTIONS
            if mode != "poly":
                skm_junctions = {} # stores the coordinates and the intersecting roads ids for each junction
                skm_lines = skm_data.loc[(skm_data['aligned']) & (skm_data['otype'] == 'Line')]
                skm_lines = skm_lines[skm_lines['id'].isin([int(id) for id in list(bsm_lines['id'])])] # considers the aligned lines only
                js_id = 0 # self-generated id for the junctions on the sketch map
                for i in range(skm_lines.shape[0]): # for each line
                    l1_id = int(skm_lines[['id']].iloc[i, 0])
                    l1_geom = skm_lines[['geometry']].iloc[i, 0]
                    for j in range(i+1, skm_lines.shape[0]): # for each other unexplored line
                        l2_id = int(skm_lines[['id']].iloc[j, 0])
                        l2_geom = skm_lines[['geometry']].iloc[j, 0]
                        inter_coords = shp.get_coordinates(l1_geom.intersection(l2_geom)).tolist()
                        if inter_coords != []: # if there is really an intersection point
                            new_junc = True # in a first time, suppose that the junction is new
                            for junc in skm_junctions.keys():
                                if inter_coords[0] == skm_junctions[junc][0]: # if this junction was already detected
                                    if l1_id in skm_junctions[junc][1] and not l2_id in skm_junctions[junc][1]: # add the missing id to the line ids
                                        skm_junctions[junc][1].append(l2_id)
                                    elif l2_id in skm_junctions[junc][1] and not l1_id in skm_junctions[junc][1]:
                                        skm_junctions[junc][1].append(l1_id)
                                    new_junc = False # finally this junction is already known
                                    continue
                            if new_junc: # if finally this junction is really new
                                skm_junctions['JS'+str(js_id)] = [inter_coords[0], [l1_id, l2_id]] # add [JSx]: [[coords], [id_lines]] in skm_junctions
                                js_id += 1
                for junc in skm_junctions.keys():
                    r_buffer = 1 # buffer around the point
                    x, y = skm_junctions[junc][0]
                    x_min, x_max, y_min, y_max = x - r_buffer, x + r_buffer, y - r_buffer, y + r_buffer
                    up_left, up_right, down_left, down_right = [x_min, y_max], [x_max, y_max], [x_min, y_min], [x_max, y_min]
                    up_center = [(x_min + x_max) / 2, y_max]
                    down_center = [(x_min + x_max) / 2, y_min]
                    left_center = [x_min, (y_min + y_max) / 2]
                    right_center = [x_max, (y_min + y_max) / 2]
                    skm_dict_mbr[junc] = np.array([up_left, up_center, up_right, right_center, down_right, down_center, down_left, left_center, up_left])
                
                for bsm_junc in bsm_junctions.keys(): # alignment of the junctions
                    bsm_id_lines = bsm_junctions[bsm_junc][1]
                    for skm_junc in skm_junctions.keys():
                        skm_id_lines = skm_junctions[skm_junc][1]
                        if set(skm_id_lines) <= set(bsm_id_lines): # if all the skm_id_lines included in bsm_id_lines (partial or complete intersection)
                            dict_align[bsm_junc] = skm_junc

            
            # ---------- PLOTS --------------------------------------------------------------------------------------------------------------------------
            """
            bsm_data.plot()
            for id in bsm_dict_mbr.keys():
                X, Y = [], []
                for point in bsm_dict_mbr[id]:
                    x, y = point
                    X.append(x)
                    Y.append(y)
                    plt.scatter(x, y, c='r', s=5)
                plt.plot(X, Y, c='r', lw=1)
                plt.text(np.mean(X), np.mean(Y), str(id), weight="bold", size="x-small")
            plt.title('Landmarks detection on the base map')
            plt.xlabel('x')
            plt.ylabel('y')
            ax = plt.gca()
            ax.set_aspect('equal')
            
            skm_data.plot()
            for id in skm_dict_mbr.keys():
                X, Y = [], []
                for point in skm_dict_mbr[id]:
                    x, y = point
                    X.append(x)
                    Y.append(y)
                    plt.scatter(x, y, c='r', s=5)
                plt.plot(X, Y, c='r', lw=1)
                plt.text(np.mean(X), np.mean(Y), str(id), weight="bold", size="x-small")
            plt.title('Landmarks detection on the sketch map')
            plt.xlabel('x')
            plt.ylabel('y')
            ax = plt.gca()
            ax.set_aspect('equal')
            """
            
            # ---------- GMDA MEASURES ------------------------------------------------------------------------------------------------------------------

            nTL = len(bsm_dict_mbr) # total number of landmarks on the base map
            nDL = len(dict_align.keys()) # number of drawn landmarks on the sketch map (nDL <= nTL)
            n_nTL = math.comb(8*nTL, 2) - nTL*math.comb(8, 2) # number of comparisons in advanced mode for nTL
            n_nDL = math.comb(8*nDL, 2) - nDL*math.comb(8, 2) # number of comparisons in advanced mode for nDL

            if nDL < 2: # not enough drawn landmarks => impossible to calculate
                print("ERROR: not enough landmarks for", loc, xx, skm_id)
                results.loc[len(results)] = [loc, xx, skm_id, 'NULL', 'NULL', 'NULL', 'NULL', 'NULL', 'NULL', 'NULL', 'NULL']
                continue
            sum_can_score = 0 # prepared for CanOrg & CanAcc
            max_D_bsm, max_D_skm = 0, 0 # prepared for ScBias & DistAcc
            sum_sin, sum_cos, sum_diff_abs = 0, 0, 0 # prepared for RotBias & AngAcc
            for i in range(nDL-1): # for every drawn landmark except the last one (all combinations will be already tested)
                k1 = list(dict_align.keys())[i]
                b1_pts = bsm_dict_mbr[k1]
                if not dict_align[k1] in skm_dict_mbr.keys(): # if sid in alignment file but not in GeoJSON
                    print("Warning: sid in alignm but not in GeoJSON for", loc, xx, skm_id)
                    continue
                s1_pts = skm_dict_mbr[dict_align[k1]]
                k_to_explore = list(dict_align.keys())[i+1:] # explore all the not yet tested comparisons
                for k2 in k_to_explore:
                    b2_pts = bsm_dict_mbr[k2]
                    if not dict_align[k2] in skm_dict_mbr.keys(): # if sid in alignment file but not in GeoJSON
                        print("Warning: id in alignm but not in GeoJSON for", loc, xx, skm_id)
                        continue
                    s2_pts = skm_dict_mbr[dict_align[k2]]
                    for j in range(8): # 8 vertices for each MBR
                        b1_x, b1_y = b1_pts[j]
                        s1_x, s1_y = s1_pts[j]
                        for l in range(8):
                            b2_x, b2_y = b2_pts[l]
                            s2_x, s2_y = s2_pts[l]
                            ## CanOrg & CanAcc: NS comparison
                            if (b1_y < b2_y and s1_y < s2_y) or (b1_y > b2_y and s1_y > s2_y):
                                sum_can_score += 1
                            ## CanOrg & CanAcc: EW comparison
                            if (b1_x < b2_x and s1_x < s2_x) or (b1_x > b2_x and s1_x > s2_x):
                                sum_can_score += 1
                            ## ScBias & DistAcc: maximum distances on the base & sketch maps
                            dist_bsm = np.sqrt( (b1_x-b2_x)**2 + (b1_y-b2_y)**2 )
                            dist_skm = np.sqrt( (s1_x-s2_x)**2 + (s1_y-s2_y)**2 )
                            if dist_bsm > max_D_bsm:
                                max_D_bsm = dist_bsm
                            if dist_skm > max_D_skm:
                                max_D_skm = dist_skm
                            ## RotBias & AngAcc: angles comparison
                            ang_bsm = np.arctan2(b2_x-b1_x, b2_y-b1_y)
                            ang_skm = np.arctan2(s2_x-s1_x, s2_y-s1_y)
                            ang_diff = ang_skm - ang_bsm
                            while ang_diff < -np.pi:
                                ang_diff += 2*np.pi
                            while ang_diff > np.pi:
                                ang_diff -= 2*np.pi
                            sum_sin += np.sin(ang_diff)
                            sum_cos += np.cos(ang_diff)
                            sum_diff_abs += np.abs( 180 / np.pi * ang_diff )

            ## ScBias & DistAcc: scale factors comparison
            sum_dr_diff, sum_dr_diff_abs = 0, 0
            for i in range(nDL-1): # for every drawn landmark except the last (all combinations already tested)
                k1 = list(dict_align.keys())[i]
                b1_pts = bsm_dict_mbr[k1]
                if not dict_align[k1] in skm_dict_mbr.keys(): # if sid in alignment file but not in GeoJSON
                    continue
                s1_pts = skm_dict_mbr[dict_align[k1]]
                k_to_explore = list(dict_align.keys())[i+1:] # explore all the not yet tested comparisons
                for k2 in k_to_explore:
                    b2_pts = bsm_dict_mbr[k2]
                    if not dict_align[k2] in skm_dict_mbr.keys(): # if sid in alignment file but not in GeoJSON
                        continue
                    s2_pts = skm_dict_mbr[dict_align[k2]]
                    for j in range(8): # 8 vertices for each MBR
                        b1_x, b1_y = b1_pts[j]
                        s1_x, s1_y = s1_pts[j]
                        for l in range(8):
                            b2_x, b2_y = b2_pts[l]
                            s2_x, s2_y = s2_pts[l]
                            dist_bsm = np.sqrt( (b1_x-b2_x)**2 + (b1_y-b2_y)**2 )
                            dist_skm = np.sqrt( (s1_x-s2_x)**2 + (s1_y-s2_y)**2 )
                            sum_dr_diff += dist_skm / max_D_skm - dist_bsm / max_D_bsm
                            sum_dr_diff_abs += np.abs( dist_skm / max_D_skm - dist_bsm / max_D_bsm )

            can_org = sum_can_score / (2*n_nTL)
            can_acc = sum_can_score / (2*n_nDL)
            sc_bias = sum_dr_diff / n_nDL
            dist_acc = 1 - sum_dr_diff_abs / n_nDL
            rot_bias = 180 / np.pi * np.arctan2(sum_sin/n_nDL, sum_cos/n_nDL)
            ang_acc = 1 - sum_diff_abs / (180 * n_nDL)

            results.loc[len(results)] = [loc, xx, skm_id, nTL, nDL, np.round(can_org, 2), np.round(can_acc, 2), np.round(sc_bias, 2), np.round(dist_acc, 2), np.round(rot_bias, 2), np.round(ang_acc, 2)]
            print("Done for", loc, xx, skm_id)
            
            print(f"CanOrg = {np.round(can_org, 2)}")
            print(f"CanAcc = {np.round(can_acc, 2)}")
            print(f"ScBias = {np.round(sc_bias, 2)}")
            print(f"DistAcc = {np.round(dist_acc, 2)}")
            print(f"RotBias = {np.round(rot_bias, 2)}°")
            print(f"AngAcc = {np.round(ang_acc, 2)}")


results.to_csv("results.csv", index=False) # stores the results in a CSV file

Done for 1 AS 1
CanOrg = 0.01
CanAcc = 0.75
ScBias = 0.02
DistAcc = 0.86
RotBias = -36.99°
AngAcc = 0.79
Done for 1 AS 2
CanOrg = 0.21
CanAcc = 0.72
ScBias = -0.03
DistAcc = 0.9
RotBias = -41.36°
AngAcc = 0.77


d:\anaconda3\envs\geospatial\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'id' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


Done for 1 AS 3
CanOrg = 0.03
CanAcc = 0.67
ScBias = -0.02
DistAcc = 0.84
RotBias = -32.46°
AngAcc = 0.82


KeyboardInterrupt: 